In [1]:
import scirpy as ir
import scanpy as sc
from glob import glob
import pandas as pd
import tarfile
import anndata
import warnings
import scanpy as sc
import anndata as an
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import math
from scipy import stats
import scipy.stats as stats

In [3]:
def compare_tests_metrics(metrics_standard, metrics_experimental, n_standard, n_experimental, margin=0.05, alpha=0.05):
    """
    Compare two tests based on sensitivity, specificity, PPV, NPV
    metrics_standard, metrics_experimental: dictionary with keys ['Sensitivity','Specificity','PPV','NPV']
    n_standard, n_experimental: total sample sizes for each test
    margin: noninferiority margin
    alpha: significance level (default 0.05)
    """
    results = {}
    metrics = ['Sensitivity','Specificity','PPV','NPV']
    
    for metric in metrics:
        p_std = metrics_standard[metric]
        p_exp = metrics_experimental[metric]
        diff = p_exp - p_std
        
        # Standard error for difference of proportions
        se = np.sqrt(p_exp*(1-p_exp)/n_experimental + p_std*(1-p_std)/n_standard)
        
        # Z-statistics
        z_superior = diff / se
        z_noninferior = (diff + margin) / se
        
        # P-values
        p_superior = 1 - stats.norm.cdf(z_superior)  # Probability experimental > standard
        p_noninferior = stats.norm.cdf(z_noninferior)  # Probability experimental within margin
        
        # Flags based on alpha
        superior_flag = "Yes" if p_superior < alpha else "No"
        noninferior_flag = "Yes" if p_noninferior < alpha else "No"
        
        results[metric] = {
            'Standard': p_std,
            'Experimental': p_exp,
            'Difference': diff,
            'SE': se,
            'P_superior': p_superior,
            'Superior': superior_flag,
            'P_noninferior': p_noninferior,
            'Noninferior': noninferior_flag
        }
    
    return results


In [4]:
# Usage
metrics_standard = {'Sensitivity':0.714, 'Specificity':0.545, 'PPV':0.50, 'NPV':0.75}
metrics_experimental = {'Sensitivity':1.0, 'Specificity':0.515, 'PPV':0.568, 'NPV':1.0} # Change these numbers based on own test.
n_standard = 54
n_experimental = 54
margin = 0.05

results = compare_tests_metrics(metrics_standard, metrics_experimental, n_standard, n_experimental, margin)

# Display results
for metric, vals in results.items():
    print(f"{metric}: {vals}")

Sensitivity: {'Standard': 0.714, 'Experimental': 1.0, 'Difference': 0.28600000000000003, 'SE': 0.06149435385102892, 'P_superior': 1.6529809238052806e-06, 'Superior': 'Yes', 'P_noninferior': 0.9999999767128348, 'Noninferior': 'No'}
Specificity: {'Standard': 0.545, 'Experimental': 0.515, 'Difference': -0.030000000000000027, 'SE': 0.0960082943947688, 'P_superior': 0.6226594611932161, 'Superior': 'No', 'P_noninferior': 0.5825086206158532, 'Noninferior': 'No'}
PPV: {'Standard': 0.5, 'Experimental': 0.568, 'Difference': 0.06799999999999995, 'SE': 0.09577906676111242, 'P_superior': 0.23886222474115737, 'Superior': 'No', 'P_noninferior': 0.8910258279639975, 'Noninferior': 'No'}
NPV: {'Standard': 0.75, 'Experimental': 1.0, 'Difference': 0.25, 'SE': 0.05892556509887896, 'P_superior': 1.1045248499264027e-05, 'Superior': 'Yes', 'P_noninferior': 0.9999998220685035, 'Noninferior': 'No'}
